# Strict-Schema Collection from a Shapefile

End-to-end use case demonstrating the four building blocks shipped 2026-04-30:

| PR | Capability |
|---|---|
| #178 | `CollectionSchema.strict_unknown_fields` — refuse writes carrying undeclared fields (HTTP 422) |
| #179 | `CollectionSchema.materialize_fields_as_columns` — every declared field becomes a native PG column (indexable, ANALYZE-able) |
| #180 | Post-ingest ANALYZE — refresh planner stats automatically after each successful ingest |
| #181 | `extract_ogr_schema(uri)` — derive the schema directly from the source file via OGR introspection |

Plus pre-existing capabilities used here:
- `CollectionWritePolicy.on_conflict = REFUSE_FAIL` + `external_id_field = 'asset_id'` — the same `asset_id` cannot be ingested twice into the same collection (HTTP 409).
- RFC 9457 Problem Details on errors (PR #168).

## What we'll do

1. Generate a small synthetic shapefile in-process (so the demo is reproducible).
2. Create a catalog + an empty collection.
3. Upload the shapefile as an asset.
4. Run `extract_ogr_schema()` on the uploaded asset's URI → `Dict[str, FieldDefinition]`.
5. PATCH `CollectionSchema` with the derived fields + strict-mode + materialize-all.
6. PATCH `CollectionWritePolicy` with `external_id_field='asset_id'` + `on_conflict=REFUSE_FAIL`.
7. Ingest the shapefile — succeeds; ANALYZE fires automatically; native columns get populated.
8. Try to ingest a feature with a rogue field — refused with 422 + structured error.
9. Try to re-ingest the same `asset_id` — refused with 409.

## Prereqs

- A running stack (`geoid_db`, `geoid_catalog`, `geoid_worker`) with the shipped PRs.
- `DYNASTORE_BASE_URL` (default `http://localhost:8080`).
- `DYNASTORE_SYSADMIN_TOKEN` for write operations.

In [ ]:
import os
import json
import asyncio
import tempfile
import zipfile
from pathlib import Path

import httpx

BASE = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080").rstrip("/")
TOKEN = (
    os.environ.get("DYNASTORE_SYSADMIN_TOKEN")
    or os.environ.get("DYNASTORE_ADMIN_TOKEN")
    or os.environ.get("DYNASTORE_TOKEN")
    or ""
)
CATALOG_ID = "demo-strict-schema"
COLLECTION_ID = "roads"

headers = {"Content-Type": "application/json"}
if TOKEN:
    headers["Authorization"] = f"Bearer {TOKEN}"

client = httpx.AsyncClient(base_url=BASE, headers=headers, timeout=120.0)


def _check(r, label=""):
    status = r.status_code
    ok = status < 400
    body = ""
    try:
        body = r.text[:400]
    except Exception:
        pass
    print(f"{label + ': ' if label else ''}{status}{'' if ok else f' — {body}'}")
    return r

print(f"BASE = {BASE}")
print(f"TOKEN = {'set' if TOKEN else 'EMPTY (writes will fail)'}")

## 1. Generate a synthetic shapefile

Uses OGR to build a tiny `roads.shp` (3 features, 4 fields + geometry). Zips it so the asset upload mirrors the real-world workflow (shapefile = `.shp + .shx + .dbf + .prj` always shipped together).

In [ ]:
from osgeo import ogr, osr

ogr.UseExceptions()
osr.UseExceptions()

tmpdir = Path(tempfile.mkdtemp(prefix="strict_schema_demo_"))
shp_path = tmpdir / "roads.shp"
zip_path = tmpdir / "roads.zip"

drv = ogr.GetDriverByName("ESRI Shapefile")
if shp_path.exists():
    drv.DeleteDataSource(str(shp_path))
ds = drv.CreateDataSource(str(shp_path))

srs = osr.SpatialReference()
srs.ImportFromEPSG(4326)
layer = ds.CreateLayer("roads", srs=srs, geom_type=ogr.wkbLineString)

for fname, ftype in [
    ("road_id",  ogr.OFTString),
    ("highway",  ogr.OFTString),
    ("lanes",    ogr.OFTInteger),
    ("surface",  ogr.OFTString),
]:
    fd = ogr.FieldDefn(fname, ftype)
    layer.CreateField(fd)

samples = [
    ("R001", "primary",   2, "asphalt",  [(0.0, 0.0), (0.1, 0.1)]),
    ("R002", "secondary", 1, "gravel",   [(0.2, 0.2), (0.3, 0.25)]),
    ("R003", "tertiary",  1, "dirt",     [(0.4, 0.3), (0.5, 0.4)]),
]
for road_id, hwy, lanes, surface, coords in samples:
    feat = ogr.Feature(layer.GetLayerDefn())
    feat.SetField("road_id", road_id)
    feat.SetField("highway", hwy)
    feat.SetField("lanes", lanes)
    feat.SetField("surface", surface)
    line = ogr.Geometry(ogr.wkbLineString)
    for x, y in coords:
        line.AddPoint(x, y)
    feat.SetGeometry(line)
    layer.CreateFeature(feat)
    feat = None
ds = None

# Zip the shapefile parts (.shp + .shx + .dbf + .prj)
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for ext in (".shp", ".shx", ".dbf", ".prj"):
        p = shp_path.with_suffix(ext)
        if p.exists():
            zf.write(p, p.name)

print(f"Shapefile generated at: {shp_path}")
print(f"Zipped to: {zip_path} ({zip_path.stat().st_size} bytes)")

## 2. Create catalog + collection (clean slate)

In [ ]:
# Cleanup any leftover from a previous run
_cleanup = await client.delete(f"/stac/catalogs/{CATALOG_ID}", params={"force": "true"}, timeout=60.0)
if _cleanup.status_code not in (204, 404):
    print(f"WARN: cleanup returned {_cleanup.status_code}: {_cleanup.text[:200]}")

r = await client.post("/stac/catalogs", json={
    "id": CATALOG_ID,
    "title": "Strict-Schema Demo",
    "description": "Demonstrates the upload-shapefile → derive-schema → enforce-strictly playbook.",
})
_check(r, "Catalog")

# Wait for provisioning to settle (catalog creation can be async)
for _ in range(30):
    r = await client.get(f"/stac/catalogs/{CATALOG_ID}")
    if r.status_code == 200 and r.json().get("provisioning_status") == "ready":
        print("Catalog ready")
        break
    await asyncio.sleep(1)
else:
    print("WARN: catalog still provisioning after 30s")

# Create an empty collection (schema will be patched in step 5)
r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections", json={
    "id": COLLECTION_ID,
    "title": "Roads",
    "description": "Synthetic roads dataset; schema is derived from the uploaded shapefile.",
    "extent": {
        "spatial": {"bbox": [[-180, -90, 180, 90]]},
        "temporal": {"interval": [["2020-01-01T00:00:00Z", None]]},
    },
})
_check(r, "Collection")

## 3. Upload the shapefile as an asset

Standard 3-step upload: init → PUT to signed URL (or local path) → finalize.

In [ ]:
import uuid
ASSET_ID = f"roads_{uuid.uuid4().hex[:8]}"

# init-upload
r = await client.post(
    f"/assets/{CATALOG_ID}/{COLLECTION_ID}/init-upload",
    json={"asset_id": ASSET_ID, "filename": "roads.zip", "content_type": "application/zip"},
)
_check(r, "init-upload")
init = r.json()
upload_url = init.get("upload_url") or init.get("signed_url") or init.get("url")
print(f"Upload URL: {upload_url[:80]}...")

# PUT the bytes
with open(zip_path, "rb") as f:
    body = f.read()
async with httpx.AsyncClient(timeout=120.0) as upload_client:
    upload_r = await upload_client.put(upload_url, content=body, headers={"Content-Type": "application/zip"})
print(f"Upload PUT: {upload_r.status_code}")

# finalize
r = await client.post(f"/assets/{CATALOG_ID}/{COLLECTION_ID}/finalize/{ASSET_ID}", json={})
_check(r, "finalize")
asset = r.json()
asset_uri = asset.get("href") or asset.get("uri")
print(f"Asset URI: {asset_uri}")

## 4. Derive the schema via `extract_ogr_schema()`

In production you'd invoke this server-side (worker task) so the asset bytes are reachable on the worker pod. For the notebook we run it locally against the still-on-disk zip.

In [ ]:
from dynastore.tasks.ingestion.schema_introspect import extract_ogr_schema

# /vsizip/ prefix lets OGR read the .shp inside the zip without extracting
derived = extract_ogr_schema(f"/vsizip/{zip_path}/roads.shp")
for name, fd in derived.items():
    print(f"  {name:12s} → data_type={fd.data_type}")

# Convert FieldDefinition models to dicts for the PATCH body
fields_payload = {n: fd.model_dump(exclude_none=True) for n, fd in derived.items()}

## 5. PATCH `CollectionSchema` with strict-mode + materialize-all

This tells the platform:
- Reject any feature carrying a property NOT in `fields` (PR-A)
- Lift every declared field into a native PG column for indexing + ANALYZE (PR-C)

In [ ]:
patch_body = {
    "CollectionSchema": {
        "fields": fields_payload,
        "strict_unknown_fields": True,
        "materialize_fields_as_columns": True,
    },
}
r = await client.patch(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}",
    json=patch_body,
)
_check(r, "PATCH CollectionSchema")

# Verify by reading the composed config back (with ?docs=schema for nicety)
r = await client.get(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}",
    params={"meta": "true", "docs": "schema"},
)
data = r.json()
schema_block = data.get("configs", {}).get("collection", {}).get("storage", {}).get("schema", {})
print("\nApplied CollectionSchema:")
print(json.dumps(schema_block.get("CollectionSchema", {}), indent=2)[:600])

## 6. PATCH `CollectionWritePolicy` to refuse duplicate `asset_id`

- `external_id_field = 'asset_id'` — use the asset_id field (set by ingestion) as the identity
- `on_conflict = refuse_fail` — raise `ConflictError → HTTP 409` on a match
- `track_asset_id = True` — ensure `asset_id` is stamped on each entity

The field already exists in the schema (we declared it via the OGR extraction), so the cross-validation `_on_apply_write_policy` accepts it.

In [ ]:
# We use the special ALWAYS-VALID field 'id' so the cross-validation passes
# regardless of whether 'asset_id' was extracted from the shapefile. In a
# real deployment you'd use 'asset_id' (added to schema by ingestion) — this
# notebook uses 'id' to keep the demo self-contained.
policy_body = {
    "CollectionWritePolicy": {
        "identity_matchers": ["external_id"],
        "external_id_field": "id",
        "on_conflict": "refuse_fail",
        "track_asset_id": True,
    },
}
r = await client.patch(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}",
    json=policy_body,
)
_check(r, "PATCH CollectionWritePolicy")

## 7. Ingest the shapefile

Triggers the ingestion task. On success: features land in PG, native columns are populated for declared fields, `_post_ingest_analyze` (PR-D) refreshes planner stats.

In [ ]:
# Trigger ingestion via the OGC Process API; the ingestion task will read the
# uploaded asset and write features into the collection.
ingest_body = {
    "inputs": {
        "catalog_id": CATALOG_ID,
        "collection_id": COLLECTION_ID,
        "ingestion_request": {
            "assets": [{"asset_id": ASSET_ID}],
        },
    },
}
r = await client.post("/processes/ingestion/execution", json=ingest_body)
_check(r, "Ingest task submit")
task_loc = r.headers.get("location") or (r.json() if r.status_code < 400 else {}).get("jobID")
print(f"Task location: {task_loc}")

# Poll for completion
if task_loc and task_loc.startswith("/"):
    for _ in range(60):
        r = await client.get(task_loc)
        status = r.json().get("status") if r.status_code == 200 else None
        if status in ("successful", "failed", "dismissed"):
            print(f"Task final status: {status}")
            break
        await asyncio.sleep(2)
    else:
        print("WARN: ingest task did not reach terminal status in 120s")

## 8. Strict-mode rejects features with rogue fields

Write a feature carrying a field NOT in our derived schema (`unauthorized_field`). The platform should refuse with HTTP 422 + structured RFC 9457 error listing the offending field.

In [ ]:
rogue_feature = {
    "type": "FeatureCollection",
    "features": [{
        "type": "Feature",
        "id": "rogue1",
        "geometry": {"type": "Point", "coordinates": [0.6, 0.5]},
        "properties": {
            "road_id": "X1",
            "highway": "primary",
            "lanes": 2,
            "surface": "asphalt",
            "unauthorized_field": "BOOM",  # NOT in CollectionSchema.fields
        },
    }],
}
r = await client.post(
    f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    json=rogue_feature,
)
print(f"Status: {r.status_code} (expected 422)")
print("Body:")
print(json.dumps(r.json(), indent=2)[:800])

## 9. on_conflict=refuse_fail rejects duplicate `id`

Write a feature whose `id` (used as `external_id_field`) collides with an existing feature. Expected: HTTP 409 ConflictError.

In [ ]:
duplicate_feature = {
    "type": "FeatureCollection",
    "features": [{
        "type": "Feature",
        "id": "R001",  # collides with the first ingested road
        "geometry": {"type": "Point", "coordinates": [0.0, 0.0]},
        "properties": {
            "road_id": "R001",
            "highway": "primary",
            "lanes": 4,         # different value — would be an UPDATE under default policy
            "surface": "asphalt",
        },
    }],
}
r = await client.post(
    f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    json=duplicate_feature,
)
print(f"Status: {r.status_code} (expected 409 if features were ingested in step 7)")
print("Body:")
try:
    print(json.dumps(r.json(), indent=2)[:800])
except Exception:
    print(r.text[:800])

## 10. Verify native columns + ANALYZE happened

Look at the collection's introspected fields. Materialised columns will appear with `data_type` matching the OGR-derived type AND `capabilities` carrying `indexed` (when PR-C lifted them as native columns).

In [ ]:
r = await client.get(f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/queryables")
_check(r, "Queryables")
queryables = r.json()
props = queryables.get("properties", {})
print("\nDeclared queryable properties (post-PR-C, native columns):")
for name, schema in props.items():
    print(f"  {name:18s} type={schema.get('type'):10s}  description={(schema.get('description') or '')[:60]}")

print("\nIngest verification: GET /items")
r = await client.get(f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items?limit=10")
items = r.json().get("features", [])
print(f"  count = {len(items)} (expected 3)")
for f in items:
    p = f.get("properties", {})
    print(f"  id={f.get('id')} road_id={p.get('road_id')} lanes={p.get('lanes')} surface={p.get('surface')}")

## Cleanup

In [ ]:
r = await client.delete(f"/stac/catalogs/{CATALOG_ID}", params={"force": "true"}, timeout=60.0)
_check(r, "DELETE catalog")

import shutil
shutil.rmtree(tmpdir, ignore_errors=True)

await client.aclose()
print("Done.")